# AI Data Analyst Demo

## Databricks Genie with LangGraph Multi-Agent Orchestration

This notebook demonstrates a multi-agent AI system that:
- Routes natural language queries to specialized agents
- Uses **Genie** for structured data analysis (NL to SQL)
- Uses **RAG** for document/policy questions
- Orchestrates with **LangGraph** and **ChatDatabricks**

## 1. Setup and Dependencies

In [ ]:
# Install dependencies (run once)
%pip install databricks-sdk>=0.40.0 databricks-langchain>=0.13.0 databricks-vectorsearch>=0.64 langgraph>=0.2.0 langchain-core>=0.3.0 langchain-text-splitters>=0.3.0 pydantic>=2.0.0 python-dotenv>=1.0.0 -q

In [ ]:
# Restart Python to pick up new packages (Databricks)
# dbutils.library.restartPython()

In [ ]:
# Add src to path for imports
import sys
import os

# For Databricks notebooks
if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    # Get the workspace path
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    workspace_path = '/Workspace' + '/'.join(notebook_path.split('/')[:-2])
    if workspace_path not in sys.path:
        sys.path.insert(0, workspace_path)
else:
    # For local development
    project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

## 2. Configuration

In [ ]:
import os
from src.config import Config, clear_config_cache

# Clear any cached config
clear_config_cache()

# Configuration - uses environment variables or Databricks widgets
# Set these before running:
#   export GENIE_SPACE_ID=your-space-id
#   export WAREHOUSE_ID=your-warehouse-id
# Or in Databricks, use notebook widgets

config = Config(
    genie_space_id=os.getenv("GENIE_SPACE_ID", ""),  # Your Genie Space ID
    warehouse_id=os.getenv("WAREHOUSE_ID", ""),      # Your SQL Warehouse ID
    model_endpoint=os.getenv("MODEL_ENDPOINT", "databricks-meta-llama-3-3-70b-instruct"),
    mock_mode=os.getenv("MOCK_MODE", "false").lower() == "true",
    # Vector Search for RAG (optional)
    vector_search_endpoint=os.getenv("VECTOR_SEARCH_ENDPOINT", ""),
    vector_search_index=os.getenv("VECTOR_SEARCH_INDEX", ""),
)

# Validate configuration
errors = config.validate()
if errors:
    print("Configuration warnings:")
    for error in errors:
        print(f"  - {error}")
else:
    print("Configuration valid!")

# Check RAG configuration
if config.is_rag_configured():
    print("RAG (Vector Search) configured!")
else:
    print("RAG running in mock mode (Vector Search not configured)")

print(f"\nGenie Space ID: {config.genie_space_id or '(not set)'}")
print(f"Mock mode: {config.mock_mode}")
print(f"Model endpoint: {config.model_endpoint}")

In [ ]:
# === CHECKPOINT: Setup Complete ===
# Save checkpoint after successful configuration
from src.demo import (
    CheckpointManager, create_checkpoint, display_checkpoint_widget,
    CHECKPOINT_SETUP, display_success, display_demo_marker, is_presenter_mode
)

# Display presenter marker (only visible in presenter mode)
display_demo_marker(3, "Configuration and setup walkthrough", "action")

# Create checkpoint manager and save setup state
checkpoint_manager = CheckpointManager()
if config and not errors:
    create_checkpoint(
        name=CHECKPOINT_SETUP,
        config=config,
        pipeline_state=None,  # No pipeline state in basic demo
        manager=checkpoint_manager,
    )
    display_success("Setup checkpoint saved!", "You can restore to this point with checkpoint_manager.restore('setup_complete')")

## 3. Initialize Agents

In [ ]:
# Presenter timing marker (only visible in presenter mode)
from src.demo import display_demo_marker
display_demo_marker(2, "Agent initialization explanation", "action")

In [ ]:
from src.agents import GenieDataAgent, RAGAgent
from src.agents.supervisor import create_simple_supervisor, SupervisorRunner

# Initialize the supervisor (includes Genie and RAG agents)
supervisor = create_simple_supervisor(config)

print("Supervisor agent initialized!")
print(f"  - Genie Agent: {'Mock' if config.mock_mode else 'Live'}")
print(f"  - RAG Agent: {'Live (Vector Search)' if config.is_rag_configured() else 'Mock'}")

## 4. Demo: Simple Data Query (TPCH)

Ask a straightforward data question about the TPCH sales data.

In [ ]:
# Simple data query - routes to Genie
question = "What are the total sales by order status?"

print(f"Question: {question}")
print("="*60)

response = supervisor.query(question, reset_history=True)
print(response)

In [ ]:
# === CHECKPOINT: Basic Query Complete ===
from src.demo import CHECKPOINT_BASIC_QUERY, create_checkpoint, display_success

# Save checkpoint after first successful query
create_checkpoint(
    name=CHECKPOINT_BASIC_QUERY,
    config=config,
    pipeline_state=None,
    extras={"last_question": question, "last_response": response[:500] if response else ""},
    manager=checkpoint_manager,
)
display_success("Query checkpoint saved!", "First successful query completed.")

## Challenge: Modify the Query

Now it's your turn! Modify the query to filter by a specific condition.

In [ ]:
# === CHALLENGE: Modify the Query ===
from src.demo import DEMO_CHALLENGES, get_challenge_runner, display_solution

# Get the challenge and runner
challenge = DEMO_CHALLENGES["modify_query"]
runner = get_challenge_runner()

# Display the challenge instructions
runner.show_challenge(challenge)

# Show solution (only visible in presenter mode)
display_solution(
    challenge.solution_code,
    "This solution filters by customer segment to narrow the results."
)

In [ ]:
# YOUR CODE HERE: Modify the question and run!
# Examples:
#   - "Show orders from AUTOMOBILE segment customers"
#   - "What is the revenue from the EUROPE region?"
#   - "Top 5 customers by order value in the BUILDING segment"

your_modified_question = "Show the top 10 customers from the AUTOMOBILE segment"  # <-- MODIFY THIS

print(f"Your Question: {your_modified_question}")
print("="*60)

# Run your modified query
your_result = supervisor.query(your_modified_question, reset_history=True)
print(your_result)

In [ ]:
# === Validate your challenge answer ===
# Pass your result to the validator

from src.demo import run_challenge

# Create a simple result object for validation
class SimpleResult:
    def __init__(self, success, data):
        self.success = success
        self.data = data

# Validate - the challenge checks if you got a successful response
# If you modified the query correctly, this should pass!
if your_result:
    result_obj = SimpleResult(success=True, data=[{"result": your_result}])
    run_challenge(challenge, result_obj)
else:
    run_challenge(challenge, None)

# If you get stuck, reveal a hint:
# runner.reveal_hint("modify_query", challenge)

## 5. Demo: Follow-up Query

Demonstrate conversation context with a follow-up question.

In [ ]:
# Follow-up question (uses conversation context)
follow_up = "Show me the top 10 customers by total order value"

print(f"Follow-up: {follow_up}")
print("="*60)

response = supervisor.query(follow_up, reset_history=True)
print(response)

## 6. Demo: Document/Policy Query

Ask about company policies - routes to RAG agent.

In [ ]:
# Policy question - routes to RAG
policy_question = "What is our refund policy?"

print(f"Question: {policy_question}")
print("="*60)

response = supervisor.query(policy_question, reset_history=True)
print(response)

## 7. Demo: Complex Multi-Agent Query

Ask a question that may require multiple agent calls or synthesis.

In [ ]:
# Complex question that might need both agents
complex_question = "What were our monthly revenue trends, and how does that align with our pricing tiers?"

print(f"Question: {complex_question}")
print("="*60)

response = supervisor.query(complex_question, reset_history=True)
print(response)

## 8. Agent Routing Demonstration

Show how different question types route to different agents.

In [ ]:
# Test questions for TPCH data
test_questions = [
    ("What is the average order value by market segment?", "Genie (Data)"),
    ("Which suppliers have the most parts?", "Genie (Data)"),
    ("Show monthly order trends", "Genie (Data)"),
    ("What are the top selling parts by revenue?", "Genie (Data)"),
]

print("Agent Routing Demonstration - TPCH Queries")
print("="*60)

for question, expected_route in test_questions:
    print(f"\nQ: {question}")
    print(f"Expected Route: {expected_route}")
    print("-"*40)
    
    response = supervisor.query(question, reset_history=True)
    # Show first 500 chars of response
    preview = response[:500] + "..." if len(response) > 500 else response
    print(preview)
    print()

## 9. Direct Agent Access (Advanced)

You can also use the agents directly without the supervisor.

In [ ]:
# Direct Genie access - query TPCH data directly
genie = GenieDataAgent(config)

result = genie.query("What are the total orders by nation?")

if result.success:
    print("Genie Query Result:")
    print(result.to_markdown_table())
    if result.sql:
        print(f"\nGenerated SQL:\n{result.sql}")
else:
    print(f"Error: {result.error}")

In [ ]:
# Direct RAG access
rag = RAGAgent(config)

result = rag.query("What security measures do we have?")

if result.success:
    print("RAG Query Result:")
    print(result.answer)
    print("\n" + result.format_sources())
else:
    print(f"Error: {result.error}")

## 10. Interactive Query Loop

Try your own questions!

In [ ]:
# Interactive query - modify this cell and run
# Try questions like:
# - "What is the revenue breakdown by region?"
# - "Show me customers in the AUTOMOBILE segment"
# - "Which parts have the highest retail price?"

your_question = "What is the revenue breakdown by region?"

print(f"Your Question: {your_question}")
print("="*60)

response = supervisor.query(your_question, reset_history=True)
print(response)

## Cleanup

In [ ]:
# Clear conversation history
supervisor.clear_history()
print("Conversation history cleared.")

---

## Architecture Summary

```
User Question
      │
      ▼
┌─────────────────────┐
│  Supervisor Agent   │  LangGraph StateGraph
│  (ChatDatabricks)   │  with function calling
└─────────┬───────────┘
          │
    ┌─────┴─────┐
    │           │
    ▼           ▼
┌────────┐  ┌────────┐
│ Genie  │  │  RAG   │
│ Agent  │  │ Agent  │
│ NL→SQL │  │ Docs   │
└────────┘  └────────┘
```

**Tools:**
- `query_data`: Routes to Genie for structured data analysis
- `search_documents`: Routes to RAG for document/policy questions

---

## Next Steps

### Continue Learning

- **Build Your Own Agent**: Hands-on workshop for building LangGraph agents
  - [03_build_your_agent.ipynb](./03_build_your_agent.ipynb)

- **Advanced Demo**: Multi-Genie orchestration with parallel queries
  - [advanced_demo.ipynb](./advanced_demo.ipynb)